![Platform+](images/banner.png)

# Lab 9: Unstructured Data + AI
## Building the Coffee Knowledge Pipeline — from raw PDFs to a customer-facing agent.

> **Persona track: Data Engineer** — same lab, your angle.

:point_right: **In this lesson, you'll build the complete production pipeline that powers the coffee shop kiosk app: parse PDFs, create a search service, deploy an agent, and test it end-to-end.**

The scenario: Management wants an AI-powered kiosk app at the counter that answers customer questions about coffee drinks. Your job is to build the data pipeline that makes it work.

We'll cover:
1. Parse all PDFs into a knowledge-base table
2. Classify documents by category
3. Explore embeddings — the primitive behind semantic search
4. Create a Cortex Search Service
5. Create a Cortex Agent backed by the search
6. Analyze a coffee video with multimodal AI
7. Build the full pipeline with cost estimation

📚 **By the end you'll be able to:**
- Build a document parsing pipeline at scale using DIRECTORY tables
- Create and query a Cortex Search Service
- Deploy a Cortex Agent on top of search
- Generate and use vector embeddings for similarity
- Process video files with multimodal AI

**Time:** ~45 min · **Lab interface: Workspace Notebook —** Cortex Code (CoCo) is your tutor — ask it to explain or fix anything.
*Source data lives in the shared, read-only `PLATPLUS_DB`; anything you create lives in your own `{{my_db}}`.*

---
👉 **Connecting to your compute service:**
1. In the top bar, if no service is connected, click to connect and select your service.
2. Set your **role and warehouse** using the picker at the top-right — or just run the setup cell below, which sets them for you in SQL. (SQL cells run on the warehouse.)
3. Once the status shows **Connected** (green), run a cell with **Shift+Enter** (run + advance) or **Cmd/Ctrl+Enter** (run this cell only); the **▶** button works too.

> 📌 The service is a container that runs the notebook kernel and your **Python** cells. **SQL cells (and Snowpark pushdown) run on the query warehouse**, not the container. It stays running until it hits its idle timeout or you suspend it manually.
> 💡 **Best practice: Start a fresh chat for this lab.** Before you begin, open a fresh CoCo chat — click **New chat** (the **+** at the top of the panel). Each lab creates its own objects, so a fresh chat ensures CoCo isn't referencing tables or schemas from a prior lab that no longer exist. Shorter conversations also produce sharper answers and use fewer tokens.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
my_wh = user + '_WH'
my_role = user + '_RL'
my_analyst_role = user + '_ANALYST_RL'
print('Context ready. Signed in as ' + user)
print('Your workspace -> db: ' + my_db + ' | wh: ' + my_wh + ' | role: ' + my_role)


## Set your session context

In [ ]:
USE ROLE {{my_role}};
CREATE WAREHOUSE IF NOT EXISTS {{my_wh}}
  WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE {{my_wh}};


In [ ]:
CREATE DATABASE IF NOT EXISTS {{my_db}};
CREATE SCHEMA IF NOT EXISTS {{my_db}}.RAW;

## 1 · Parse all documents at scale
The first step in any unstructured data pipeline: convert raw files into queryable text. We'll use [`DIRECTORY` tables](https://docs.snowflake.com/en/user-guide/data-load-dirtables) + [`AI_PARSE_DOCUMENT`](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document) to process all 8 coffee PDFs in a single statement.

**What is a `DIRECTORY` table?** A stage stores files, but a stage is not queryable on its own. Enabling a directory on a stage adds an implicit metadata layer — one queryable row per file, exposing `RELATIVE_PATH`, `SIZE`, and `LAST_MODIFIED`. Querying `DIRECTORY('@stage')` (a table function) is what lets you drive a set-based `INSERT` across every file, rather than looping one file at a time. It is the file inventory your pipeline reads from.

**About `AI_PARSE_DOCUMENT`:**

This Cortex AI function extracts text, layout, and images from documents on a stage. It supports two modes:

| Mode | Best for | What it returns |
|------|----------|----------------|
| **LAYOUT** | Most use cases — PDFs with headings, tables, paragraphs | Preserves reading order, headers, tables, and structural elements. Returns a `content` field with full extracted text. |
| **OCR** | Scanned images, photos of documents, handwritten text | Optical character recognition — extracts text from image-based documents where text isn't embedded. |

**Key output fields** (VARIANT object):
- `content` — the full extracted text in reading order
- `images` — extracted embedded images (base64-encoded), useful for multimodal RAG
- `layout` — structural metadata (headers, paragraphs, tables)

We use **LAYOUT mode** here since our PDFs have embedded text with structured headings and paragraphs.

In [ ]:
-- Inventory what's in the stage
SELECT RELATIVE_PATH, SIZE, LAST_MODIFIED
FROM DIRECTORY('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE')
WHERE RELATIVE_PATH LIKE 'coffee_kb/%.pdf'
ORDER BY RELATIVE_PATH;


Now the core step: parse **every** PDF the directory query just listed, in a single set-based `INSERT`. For each file we derive a `drink_id` from the filename, keep the source path for lineage, and store the extracted text in `content`. One statement fans out across all 8 documents — no per-file loop — which is the whole point of pairing a `DIRECTORY` table with `AI_PARSE_DOCUMENT`.

> 📌 **Timing varies.** AI function execution depends on current inference capacity — the same cell might complete in seconds or take several minutes depending on demand. If it's slow, let it run; it will complete.

In [ ]:
-- Parse ALL PDFs into a knowledge-base table
CREATE OR REPLACE TABLE {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE (
    drink_id STRING,
    source_file STRING,
    content STRING,
    category STRING,
    parsed_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

INSERT INTO {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE (drink_id, source_file, content)
SELECT
  REPLACE(SPLIT_PART(RELATIVE_PATH, '/', -1), '.pdf', '') AS drink_id,
  RELATIVE_PATH AS source_file,
  AI_PARSE_DOCUMENT(
    TO_FILE('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE', RELATIVE_PATH),
    {'mode': 'LAYOUT'}
  ):content::STRING AS content
FROM DIRECTORY('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE')
WHERE RELATIVE_PATH LIKE 'coffee_kb/%.pdf';


> :warning: **Your parse results may differ slightly from your neighbour's — that's normal.**
>
> `AI_PARSE_DOCUMENT` is primarily an extraction function (not generation), so output is mostly stable. However, minor differences in paragraph breaks, whitespace, or heading detection can occur between runs. As long as the drink names, topics, and key facts are present, your results are correct.

> :bulb: **Pipeline pattern:** In production, you'd wrap this INSERT in a Task or Stream that fires when new PDFs land in the stage. The DIRECTORY table acts as your file inventory.

In [ ]:
-- Verify
SELECT drink_id, LENGTH(content) AS content_chars, parsed_at
FROM {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE
ORDER BY drink_id;


## 2 · Classify documents by category
Use [`AI_CLASSIFY`](https://docs.snowflake.com/en/sql-reference/functions/ai_classify) to automatically categorize each drink into serving types.

If you have worked through the earlier labs, this family of AISQL functions should feel familiar — they all follow the same shape: hand the function your text plus a bit of configuration, and it returns a structured result you navigate with `:field` and cast. `AI_CLASSIFY` is one of the simplest: give it the content and a fixed list of candidate labels, and it returns the best-fit label. No prior setup, no model to train.

**Why this matters (data engineering perspective):**
- **Filtered search & partitioning** — A category column lets downstream consumers filter queries (e.g. "show me only hot drinks") without scanning all rows. In larger datasets, this becomes a clustering key candidate (a clustering key controls how Snowflake physically organizes data on disk, making filtered queries dramatically faster).
- **Data quality gate** — Classification acts as a validation step: if a document can't be classified into your known categories, it may be malformed or off-topic. Flag those for review.
- **Decoupling enrichment from ingestion** — By storing categories as a column (not embedded in the content), you can re-classify later with a better model or updated category list without re-parsing the source PDFs.

**Best practice:** Define your category list explicitly (don't let the model invent categories). This keeps values consistent and queryable — like a controlled list that prevents typos or unexpected values from creeping into your data.

In [ ]:
-- Classify each drink as hot, iced, or both
UPDATE {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE
SET category = AI_CLASSIFY(
  content,
  ['Hot Drink', 'Cold Drink', 'Both Hot and Cold']
):labels[0]::STRING;


In [ ]:
-- Check classifications
SELECT drink_id, category
FROM {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE
ORDER BY category, drink_id;


> :mag_right: **Verify.** Cold brew should come back as "Cold Drink". The rest will vary — several may land on "Hot Drink" and others on "Both Hot and Cold" depending on how each PDF reads, and that variation is expected rather than an error. In production, a document that lands in the wrong bucket is a signal — usually that your label list needs tuning (labels too broad, too narrow, or overlapping), or that the source document itself is off-topic and should be flagged for review. You would not hand-fix individual rows; you would fix the labels or the upstream data and re-run.

## 3 · Embeddings: the primitive behind semantic search
Before we let Snowflake build a search service for us, let's look at the primitive it runs on: **vector embeddings**. An embedding is a list of numbers (a vector) that captures the *meaning* of a piece of text. Text with similar meaning produces vectors that point in similar directions, so you can measure similarity with simple vector math — no LLM call, no keyword overlap required.

We'll use [`AI_EMBED`](https://docs.snowflake.com/en/sql-reference/functions/ai_embed) with Snowflake's `snowflake-arctic-embed-m-v1.5` model (768 dimensions, runs natively in Cortex — no data leaves your account) and [`VECTOR_COSINE_SIMILARITY`](https://docs.snowflake.com/en/sql-reference/functions/vector_cosine_similarity), which scores two vectors from 0 (unrelated) to 1 (identical).

The cell below embeds each drink's text on the fly and ranks every drink by similarity to espresso. Nothing is stored — this is a one-cell demonstration of the idea.

In [ ]:
-- Embed on the fly and rank drinks by similarity to espresso (no stored column)
WITH base AS (
  SELECT AI_EMBED('snowflake-arctic-embed-m-v1.5', content) AS emb
  FROM {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE
  WHERE drink_id = 'espresso'
)
SELECT
  kb.drink_id,
  ROUND(VECTOR_COSINE_SIMILARITY(
    AI_EMBED('snowflake-arctic-embed-m-v1.5', kb.content), base.emb), 3) AS similarity
FROM {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE kb, base
WHERE kb.drink_id != 'espresso'
ORDER BY similarity DESC;


> :mag_right: **What to notice.** The drinks most similar in preparation and description to espresso — the other espresso-based drinks (americano, flat white, latte, cappuccino) — should cluster near the top, while drinks made very differently (cold brew, pour over) tend to sit lower. The exact ordering will vary between runs: embedding similarity is approximate and depends on the parsed text, so read the result as a rough grouping, not a fixed leaderboard. This is pure vector math — no LLM was called to "reason" about the drinks.

**Where this goes next:** embedding-and-comparing every row by hand doesn't scale, and it only covers *doc-to-doc* similarity. In the next section you'll create a **Cortex Search Service** — and the key insight is that it does this embedding step **for you, automatically**, then adds keyword matching and reranking on top so you can query with natural language. You rarely embed by hand in production; you reach for embeddings directly only for capabilities Search doesn't expose, like the "find similar drinks" recommendation above.

## 4 · Create a Cortex Search Service
A [Cortex Search Service](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview) indexes your text data and provides semantic search (understanding meaning, not just matching keywords). This is what powers the Coffee Kiosk app's natural-language lookup.

**Why this matters (data engineering perspective):**
- **Decouples retrieval from application logic** — The search service handles indexing, ranking, and relevance scoring. Your app just sends a query string and gets results back. No need to build and maintain your own search infrastructure.
- **Automatic index refresh** — The `TARGET_LAG` parameter controls how quickly new/updated rows become searchable. You don't need to manually rebuild indexes when data changes.
- **Filterable attributes** — The `ATTRIBUTES` clause lets consumers filter results by metadata columns (like category) without building separate filtered indexes.

**Use cases for data engineers:**
| Use Case | What's indexed | Why semantic search beats keyword |
|----------|---------------|-----------------------------------|
| Customer-facing Q&A (this lab) | Product descriptions, menus | Customers ask natural questions ("something sweet") not exact terms |
| Internal knowledge base | Policy docs, runbooks, SOPs | Engineers search by intent ("how to handle timeout errors") not doc titles |
| Support ticket routing | Historical tickets + resolutions | Match new tickets to similar past issues even when phrased differently |
| Compliance & audit | Contracts, regulatory filings | Find relevant clauses without knowing the exact legal terminology |

**Statement breakdown:**
| Clause | What it does |
|--------|-------------|
| `ON content` | The text column to index for semantic search |
| `ATTRIBUTES category` | Metadata columns available for filtering at query time |
| `WAREHOUSE = {{my_wh}}` | Compute used to build and refresh the index |
| `TARGET_LAG = '1 hour'` | Maximum delay before new rows become searchable |
| `AS (SELECT ...)` | The source query — defines what data feeds the index |

**Incremental by design:** When new documents are added to your knowledge base table, the search service **automatically picks them up** on the next refresh cycle (controlled by `TARGET_LAG`). You don't rebuild the index or recreate the service — just INSERT new rows and they become searchable within the lag window. This is why the search service is a managed object, not a one-off query.

**Best practice:** Set `TARGET_LAG` based on your freshness requirements vs. cost. A 1-hour lag is good for content that changes infrequently (like menu PDFs). For real-time inventory, you'd use a shorter lag — but shorter lag = more frequent compute = higher cost.


In [ ]:
CREATE OR REPLACE CORTEX SEARCH SERVICE {{my_db}}.RAW.COFFEE_SEARCH_SVC
  ON content
  ATTRIBUTES category
  WAREHOUSE = {{my_wh}}
  TARGET_LAG = '1 hour'
  AS (
    SELECT
      drink_id,
      content,
      category
    FROM {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE
  );


### Test the search service
Query it with natural language — it returns the most relevant documents.

> :warning: **`SEARCH_PREVIEW` is for testing only — not for production use.**
>
> The function we use below (`SNOWFLAKE.CORTEX.SEARCH_PREVIEW`) is a convenience function designed for ad-hoc testing in worksheets and notebooks. It is **not performant at scale** and should never be called from an application or automated pipeline.
>
> **In production, you would:**
> - Call the search service via the **REST API** or **Python SDK** from your application layer
> - Use a **Cortex Agent** (which we build in Section 5) that handles retrieval internally
> - Query from a **Streamlit app** using the `snowflake.cortex` Python package
>
> Think of `SEARCH_PREVIEW` like a `SELECT *` with no WHERE clause — fine for a quick sanity check, not something you'd put behind a customer-facing endpoint.

Let's ask the service a real question: **"Which drink is lowest in caffeine?"** The query below sends that natural-language question to the service and formats the top 3 matches as a table. Read the results with the caveat in mind that follows.

In [ ]:
-- Search for drinks that match a question (formatted as a table)
SELECT
    f.value:"drink_id"::STRING AS drink_id,
    LEFT(f.value:"content"::STRING, 200) AS content_preview
FROM TABLE(FLATTEN(
    PARSE_JSON(SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        '{{my_db}}.RAW.COFFEE_SEARCH_SVC',
        '{
            "query": "Which drink is lowest in caffeine?",
            "columns": ["drink_id", "content"],
            "limit": 3
        }'
    )):results
)) f;


> :bulb: **How it works — hybrid retrieval, not question-answering.** Cortex Search is not pure semantic search and it is not pure keyword search — it runs **both**: vector search for semantically similar text, keyword search for lexically matching terms, and a semantic reranking step to order the final results. That is why "lowest caffeine" surfaces caffeine-related drinks even when the PDFs never use that exact phrase. The `TARGET_LAG` of 1 hour means new documents become searchable within an hour of landing in the table.
>
> :warning: **Retrieval ≠ reasoning.** Notice that cold brew may appear in the results even though it is *high* in caffeine. This is a common point of confusion: Cortex Search is a **retrieval** system, not a **question-answering** system. It finds documents semantically and lexically related to your query terms — here, anything that talks about *caffeine* — it does not reason about which drink is actually *lowest*. Getting a reasoned answer ("the lowest-caffeine drink is X") is the job of the **Agent** you build next, which sends these retrieved documents to an LLM to reason over.

> :warning: **About LLM non-determinism — important from this point forward.**
>
> Starting with the Agent and [`AI_COMPLETE`](https://docs.snowflake.com/en/sql-reference/functions/ai_complete) cells below, you'll see **free-form text generation**. Unlike the extraction and search steps above, generative AI is **non-deterministic** — even with the same model, prompt, and data, each run produces slightly different output.
>
> **Why?** Models use **temperature** and **top-k sampling** during generation. These introduce controlled randomness so responses sound natural rather than robotic. The same input can follow different valid paths through the model's probability space.
>
> **How much variation to expect by function:**
>
> | Function | Variability | Why |
> |----------|-------------|-----|
> | `AI_PARSE_DOCUMENT` | **Low** | Extraction — reads what's in the PDF. You already ran this above. |
> | `AI_CLASSIFY` | **Low–Medium** | Constrained to your category list, but confidence scores may shift slightly. |
> | Cortex Search Service | **Low** | Retrieval is mostly deterministic — same query returns same ranked results. |
> | `AI_COMPLETE` | **High** | Free-form generation — wording, structure, and detail vary each run. |
> | Cortex Agent | **High** | Combines retrieval + generation — both context selection and answer text can differ. |
> | `AI_EMBED` | **None** | Deterministic — same input always produces the same vector. |
>
> **What "correct" looks like from here:** Your Agent and AI_COMPLETE answers should cover the same *facts* from the knowledge base, but may be worded completely differently from a neighbour's output. That's expected — focus on whether the answer is grounded in the source data, not on matching exact phrasing.

## 5 · Create a Cortex Agent
A [Cortex Agent](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents) combines a search service with an LLM to provide conversational answers. This is the brain behind the Coffee Kiosk app.

**Why this matters (data engineering perspective):**
- **Centralizes the [RAG](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview) pattern** — Without an Agent, every application that wants Q&A over your data must implement its own retrieve-then-generate logic. An Agent encapsulates this: one object, one endpoint, consistent behaviour.
- **Grounded answers** — The Agent is *designed* to answer from your indexed data (the search service), which **reduces** hallucination (it does not eliminate it). When the answer isn't in your knowledge base, a well-instructed agent should say so rather than guess — which is exactly why the response guardrail matters.
- **No prompt engineering in the app** — The application just sends a user question. The Agent handles retrieval, context assembly, and generation internally. This separates data infrastructure (your job) from UI (the app developer's job).

**Statement breakdown:**
| Part | What it does |
|------|-------------|
| `CREATE OR REPLACE AGENT` | Creates a managed Agent object in your schema (like a table or view — it's a first-class Snowflake object) |
| `COMMENT = '...'` | Describes the agent's purpose — useful for discoverability and governance |
| `FROM SPECIFICATION $$ ... $$` | YAML block that defines the agent's configuration (tools, resources, behaviour) |
| `tools:` section | Lists what the agent can use — here, a `cortex_search` tool that queries your search service |
| `tool_spec.type` | The tool type — `cortex_search` connects to a Cortex Search Service |
| `tool_spec.name` | A reference name (used to link the tool to its resource below) |
| `tool_spec.description` | Tells the LLM when to use this tool — important for agents with multiple tools |
| `tool_resources:` section | Configures each tool — points the search tool to a specific service and sets parameters like `max_results` |

**Key features:**
- **Multi-tool support** — An agent can have multiple tools and the LLM decides which to call based on the question. The common ones: a **Cortex Search** tool (retrieves from unstructured text — what we use here), a **Cortex Analyst** tool (generates SQL over structured data from a semantic view, for "what were sales last quarter" style questions), and **custom tools** (stored procedures or UDFs that run your own business logic or call an external system). One agent can combine all three and route each question to the right one.
- **Versioned specifications** — The YAML spec can be updated with `ALTER AGENT` without rebuilding downstream apps
- **Callable via REST API or SQL** — Applications call the agent through the `agent:run` **REST API** (Snowflake's recommended path for apps). From SQL you can call it with [`SNOWFLAKE.CORTEX.DATA_AGENT_RUN`](https://docs.snowflake.com/en/sql-reference/functions/data_agent_run-snowflake-cortex) — a utility wrapper around that same API — which is exactly what the cell below uses

**Best practice:** Keep your Agent pointed at a single, well-scoped search service. If you need multiple knowledge domains (e.g. menu + store policies + allergens), create separate search services and list them all as separate tools in the specification — the Agent will retrieve from the most relevant one per query.

In [ ]:
CREATE OR REPLACE AGENT {{my_db}}.RAW.{{user}}_COFFEE_MENU_AGENT
  PROFILE = '{"display_name": "Coffee Menu Expert ({{user}})"}'  -- includes your username so you can find it in CoWork
  COMMENT = 'Coffee menu expert that answers customer questions about drinks'
  FROM SPECIFICATION
  $$
  tools:
    - tool_spec:
        type: "cortex_search"
        name: "CoffeeSearch"
        description: "Searches the coffee knowledge base for drink information, preparation methods, and menu details"

  tool_resources:
    CoffeeSearch:
      name: "{{my_db}}.RAW.COFFEE_SEARCH_SVC"
      max_results: "5"
  $$;


### Test the agent — the serving layer of your pipeline

The agent is the **culmination of the pipeline you just built**: Parse → Classify → Index → **Agent**. As the data engineer, you own this entire chain. Building and validating the agent is your responsibility — you're ensuring the serving layer works before handing it to the app team or the platform admin.

**Why this is your job (not the app developer's or admin's):**
- You know what data went into the search service and what quality it has
- You can validate that the agent returns grounded answers (not hallucinations)
- When something breaks downstream, you need to isolate whether it's a data problem, an indexing problem, or a generation problem

**The cell below** calls the agent via SQL using `SNOWFLAKE.CORTEX.DATA_AGENT_RUN`:
1. Sends a natural-language question as a JSON message
2. The agent retrieves relevant documents from the search service
3. Generates a grounded answer using an LLM
4. We parse the JSON response and extract just the readable text using `LATERAL FLATTEN`

**Pipeline use case — automated validation after refresh:**

In production, after your Task refreshes the knowledge base with new PDFs, you'd call the agent with a known test question and verify the answer still mentions expected keywords. This is a **serving-layer smoke test** — the same way you'd validate a table has rows after a load, you validate the agent still returns sensible answers after a data change. See the **"Agent Smoke Test"** cell further below for a working example of this pattern.

In [ ]:
-- Ask the agent a question (extract just the answer text)
WITH raw_response AS (
  SELECT TRY_PARSE_JSON(
    SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
      '{{my_db}}.RAW.{{user}}_COFFEE_MENU_AGENT',
      $${
        "messages": [
          {
            "role": "user",
            "content": [
              { "type": "text", "text": "What is the difference between a flat white and a latte? Which is stronger?" }
            ]
          }
        ]
      }$$,
      TRUE
    )
  ) AS response
)
SELECT f.value:"text"::STRING AS answer
FROM raw_response, LATERAL FLATTEN(input => response:"content") f
WHERE f.value:"type"::STRING = 'text';


### Pipeline validation — smoke test the serving layer

Here's the pattern a data engineer would use in production. After new PDFs are parsed and classified into the knowledge-base table, the Cortex Search Service **refreshes itself automatically** within its `TARGET_LAG` window — you do not rebuild or "re-index" anything by hand. Once that refresh window has passed and the new content is searchable, run a **known test question** through the agent and verify the answer contains the keywords you expect. The point of the smoke test is to catch the case where the pipeline ran but the *answer quality* silently regressed.

This is the same concept as asserting row counts after a table load — but for the AI serving layer. If the agent suddenly stops mentioning "espresso" when asked about espresso, something broke upstream (bad parse, missing doc, index lag).

**In production**, this would run inside a Task that fires after the knowledge base refresh, logging results to an audit table. Here we run it manually to demonstrate the pattern.

> :bulb: **This is just one example.** The same pattern — calling an agent as a validation or automation step within a pipeline — applies to many workflows in your organization: verifying documentation agents after wiki updates, testing support-ticket routing after model retraining, or confirming compliance agents still flag the right clauses after policy changes. The point is to see that agents aren't just customer-facing endpoints — they can be integrated into data engineering workflows wherever you need natural-language validation or automated reasoning over unstructured content.
>
> **Going further than a keyword check:** the `CONTAINS(...)` smoke test here is deliberately simple. For systematic quality measurement, Snowflake provides **Cortex Agent Evaluations** and **AI Observability** — you define a set of test questions with expected answers and Snowflake scores the agent's responses (relevance, groundedness, and more) so you can track quality over time and across model or data changes. The keyword smoke test is the quick in-pipeline guardrail; formal evaluations are the deeper, offline complement.

In [ ]:
-- Pipeline smoke test: verify agent still returns grounded answers after a data refresh
WITH test_response AS (
  SELECT TRY_PARSE_JSON(
    SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
      '{{my_db}}.RAW.{{user}}_COFFEE_MENU_AGENT',
      $${ "messages": [{"role": "user", "content": [{"type": "text", "text": "What is espresso?"}]}] }$$,
      TRUE
    )
  ) AS response
),
answer AS (
  SELECT f.value:"text"::STRING AS text
  FROM test_response, LATERAL FLATTEN(input => response:"content") f
  WHERE f.value:"type"::STRING = 'text'
)
SELECT
  text AS agent_answer,
  CONTAINS(LOWER(text), 'espresso') AS mentions_espresso,
  CONTAINS(LOWER(text), 'concentrated') OR CONTAINS(LOWER(text), 'pressure') AS mentions_key_facts,
  CASE
    WHEN CONTAINS(LOWER(text), 'espresso') 
      AND (CONTAINS(LOWER(text), 'concentrated') OR CONTAINS(LOWER(text), 'pressure'))
    THEN 'PASS'
    ELSE 'FAIL — agent may not be grounded in knowledge base'
  END AS smoke_test_result
FROM answer;


> :mag_right: **Compare:** The agent’s answers should be grounded in your PDF content. It shouldn’t hallucinate information not in the knowledge base. If it does, that’s a signal to improve your document coverage.

> 🧑‍💻 **Your job stops here.** As the data engineer, you built the full pipeline (parse → classify → index → agent) and validated it returns grounded answers. **Deploying** the agent to end-users via CoWork, controlling who can access it, and monitoring its cost — that's the Platform Admin's responsibility. See the Platform Admin notebook for that side of the story.

## 6 · Multimodal: Analyze a coffee video
Multimodal AI can process video, images, and audio — not just text. Here we use `AI_COMPLETE` with `gemini-3.1-pro` (a model that understands both visual and text input) to analyze a barista making a latte.

**Why this matters (data engineering perspective):**
- **Unstructured data isn't just PDFs** — Production pipelines increasingly need to handle images (product photos, receipts), video (training footage, security cameras), and audio (call recordings). The same stage-based pattern you used for PDFs works for all media types.
- **Metadata extraction at scale** — Instead of humans manually tagging videos, AI can extract structured metadata (what drink is being made, what equipment is used, quality scores) and write it to a table for downstream analytics.
- **Quality control & compliance** — Analyze barista training videos to verify procedures are followed correctly, or scan product images to detect labelling errors — all automated within your existing Snowflake pipeline.

**Use cases for data engineers:**
| Use Case | Input | What AI extracts |
|----------|-------|------------------|
| Training compliance | Barista videos on stage | Steps followed, technique quality score |
| Product catalog enrichment | Product photos | Colour, size, ingredients visible on label |
| Customer feedback | Audio recordings | Sentiment, topics mentioned, complaints |
| Document digitization | Scanned receipts/forms | Line items, totals, dates (beyond what OCR alone provides) |

**Best practice:** Process media files asynchronously (via Tasks) since video/image analysis is slower and more expensive than text. Store the AI-extracted insights as structured columns alongside a reference to the source file — never discard the original.

### What the SQL below does:

The cell calls `AI_COMPLETE` with a **multimodal model** (`gemini-3.1-pro`) and passes it a video file from a Snowflake stage.

**Statement breakdown:**
| Part | What it does |
|------|-------------|
| `AI_COMPLETE('gemini-3.1-pro', ...)` | Calls the LLM — but with a model that can understand both text and visual/video input |
| [`TO_FILE`](https://docs.snowflake.com/en/sql-reference/functions/to_file)`('@...DOCS_STAGE', 'coffee_videos/...')` | Converts a staged file into a FILE object the model can process |
| The text prompt | Tells the model *what* to extract from the video (drink identification, steps, quality rating) |

**Why `gemini-3.1-pro`?**
- It's one of the few models available in Snowflake Cortex that supports **multimodal input** (text + images + video)
- Text-only models (like `llama3.1-70b`) cannot process files — they'd ignore the video entirely
- For video analysis, you need a model specifically designed to understand visual frames over time

**What to expect:** The model watches the video, identifies the drink being prepared, describes the barista's steps, and rates the overall drink presentation. Output will be a text description — not structured data — because video analysis is generative.

### Play the video first
Before we ask AI anything about it, let's watch the video ourselves so we know what to expect. The cell below downloads the video file into the runtime and renders it inline for playback.

In [ ]:
# Download the video into the runtime and render it inline for playback
from IPython.display import display, HTML
import base64

session.file.get('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE/coffee_videos/coffee_making_and_review.mp4', '/tmp/')

with open('/tmp/coffee_making_and_review.mp4', 'rb') as f:
    video_bytes = f.read()

video_b64 = base64.b64encode(video_bytes).decode()
html = (
    f'<h3>Coffee Video</h3>'
    f'<video width="320" controls>'
    f'<source src="data:video/mp4;base64,{video_b64}" type="video/mp4">'
    f'</video>'
)
display(HTML(html))

> :mag_right: **Watch the video.** Note what drink is being made and how it's presented. In a moment we'll have a multimodal model describe the same video — then you can compare its description against what you just saw.

In [ ]:
-- Analyze a coffee preparation video (strip JSON quotes for readable output)
SELECT TRIM(AI_COMPLETE(
  'gemini-3.1-pro',
  'Watch this video of a barista making a coffee drink. Describe: 1) What drink is being made? 2) What steps does the barista follow? 3) Rate the overall drink presentation on a scale of 1-10.',
  TO_FILE('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE', 'coffee_videos/coffee_making_and_review.mp4')
), '"') AS video_analysis;


### Pipeline pattern — store video insights for downstream use

The query above is an ad-hoc analysis. In a real pipeline, you'd **process video files and store the extracted insights as structured data** in a table — making them queryable, joinable, and available to downstream processes.

> :bulb: **This is just one use case.** The same pattern — process media, extract structured metadata, store in a table — applies to many workflows: analyzing barista training videos for compliance, scanning product photos for label accuracy, processing customer video reviews for sentiment. The point is that unstructured media becomes structured, queryable data in your pipeline.

The cell below shows what this looks like: create a table, analyze the video, and INSERT the results as structured columns that any downstream process can consume.

**Taking it further — the full composable architecture:**

Once video insights are in a table, you can layer the same services you built earlier in this lab on top of it:

```
Raw Video → AI_COMPLETE (extract) → VIDEO_INSIGHTS table → Cortex Search Service → Cortex Agent
```

This is the exact same pattern as the PDF pipeline:
```
Raw PDFs → AI_PARSE_DOCUMENT (extract) → COFFEE_KNOWLEDGE_BASE table → Cortex Search Service → Cortex Agent
```

The building blocks are interchangeable. You could create a search service over `VIDEO_INSIGHTS`, add it as a second tool in your agent's specification, and suddenly the Coffee Kiosk app can answer questions from *both* document knowledge and video-extracted insights — all through the same conversational interface. Every section of this lab is a composable piece of infrastructure, not an isolated exercise.

> :bulb: **Why `AI_COMPLETE` here, and not `AI_TRANSCRIBE` + `AI_EXTRACT`?** Those are the right tools when the information lives in **speech** — the Data Analyst lab transcribes a spoken review (`AI_TRANSCRIBE` turns the audio track into text) and then structures that text with `AI_EXTRACT`. But `AI_TRANSCRIBE` only *listens*; it cannot *see*. Our question here is **visual** — what drink is being made, what steps are visible, how it's presented — so we need a multimodal model that watches the video frames. That is `AI_COMPLETE` with `gemini-3.1-pro`. Rule of thumb: **speech → `AI_TRANSCRIBE`; pixels/frames → `AI_COMPLETE` (multimodal).**

In [ ]:
-- Pipeline pattern: store video analysis as structured data for downstream use
CREATE TABLE IF NOT EXISTS {{my_db}}.RAW.VIDEO_INSIGHTS (
    source_file STRING,
    analyzed_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    analysis STRING
);

INSERT INTO {{my_db}}.RAW.VIDEO_INSIGHTS (source_file, analysis)
SELECT
  'coffee_videos/coffee_making_and_review.mp4',
  TRIM(AI_COMPLETE(
    'gemini-3.1-pro',
    'Watch this video. Extract: 1) Drink being made 2) Steps followed 3) Overall presentation rating 1-10. Be concise.',
    TO_FILE('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE', 'coffee_videos/coffee_making_and_review.mp4')
  ), '"');

-- Verify what was stored
SELECT * FROM {{my_db}}.RAW.VIDEO_INSIGHTS;


## 7 · Cost estimation
Before promoting this pipeline to production, estimate the token budget.

**Why this matters (data engineering perspective):**
- **Budget forecasting** — Cortex AI functions are billed by tokens consumed. Before deploying, you need to estimate costs so finance and management aren't surprised by the bill.
- **Pipeline sizing** — Token counts determine whether you can process everything in one batch or need to break it into chunks (models have context window limits).
- **Incremental vs. full refresh trade-off** — If re-processing 8 docs costs pennies but re-processing 8,000 costs dollars, that's the justification for incremental processing — only touch files that are actually new. The pipeline Task you build next does exactly this with a `NOT IN` check on `source_file`, so each run parses only PDFs that aren't already in the table.

**What the SQL below does:**

| Column | What it measures |
|--------|------------------|
| `total_docs` | Number of documents in the knowledge base |
| `embed_tokens` | Total tokens consumed if you embed all documents (one-time ingestion cost) |
| `llm_tokens` | Total tokens if all documents were passed to an LLM (e.g., for summarization or classification) |
| `avg_tokens_per_doc` | Average tokens per document — useful for estimating costs as the collection grows |

**How `AI_COUNT_TOKENS` works:**
- First argument: the **function name** (e.g., `'ai_embed'`, `'ai_complete'`) — different functions tokenize differently
- Second argument: the **model name** — different models have different tokenizers
- Third argument: the **text** to count
- It returns an integer token count *without* actually calling the model — so it's cheap to run

**Best practice:** Run cost estimation before *and* after scaling. A pipeline that costs $0.02 for 8 documents might cost $200 for 80,000. Use these numbers to justify incremental processing, choose smaller/cheaper models where quality allows, and set alerts on token consumption.

> :warning: **What this estimate does and does not cover.** `AI_COUNT_TOKENS` gives you the **text-token** cost of the token-billed functions — here, embedding text and passing text to an LLM (e.g. classification). Two costs in this lab are billed differently and are **not** captured by the query below: `AI_PARSE_DOCUMENT` is billed **per page**, and the multimodal **video** `AI_COMPLETE` (`gemini-3.1-pro`) is billed on multimodal input, not simple text tokens. Treat the number below as the text-processing slice of the bill, and size the per-page and video costs separately from their own usage.

**Use cases:**
| Use Case | What you're estimating |
|----------|------------------------|
| New pipeline deployment | Total cost of initial backfill (embed all existing docs) |
| Scaling assessment | Cost if document count grows 10x or 100x |
| Model selection | Compare token costs between different models for the same task |
| Refresh frequency planning | Daily vs. hourly refresh — how many tokens per cycle? |

In [ ]:
-- Estimate token usage for embedding and LLM processing
SELECT
  COUNT(*) AS total_docs,
  SUM(AI_COUNT_TOKENS('ai_embed', 'snowflake-arctic-embed-m-v1.5', content)) AS embed_tokens,
  SUM(AI_COUNT_TOKENS('ai_complete', 'llama3.1-70b', content)) AS llm_tokens,
  ROUND(AVG(AI_COUNT_TOKENS('ai_complete', 'llama3.1-70b', content)), 0) AS avg_tokens_per_doc
FROM {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE;


> :mag_right: **How to read these results:**
>
> *Note: The numbers below are approximate examples. Your results may differ slightly if `AI_PARSE_DOCUMENT` extracted slightly different text in Section 1. Token counting itself is deterministic (same text = same count), but the source text can vary between runs. Focus on the magnitude and pattern, not exact values.*
>
> | Column | Example result | What it means |
> |--------|------------|---------------|
> | `TOTAL_DOCS` | 8 | You have 8 documents in the knowledge base |
> | `EMBED_TOKENS` | ~3,900 | Embedding all 8 docs consumed about 3,900 tokens — this is a one-time cost at ingestion |
> | `LLM_TOKENS` | ~3,960 | If you passed all content to an LLM (e.g., for classification or summarization), it would consume about 3,960 tokens |
> | `AVG_TOKENS_PER_DOC` | ~495 | Each document averages about 495 tokens — well within a single model context window |
>
> **Putting it in perspective:**
> - Most Cortex models have context windows of 4,000–128,000 tokens, so ~495 tokens/doc is small
> - At ~$0.001–$0.003 per 1K tokens (depending on model), 8 docs costs fractions of a cent
> - But multiply by 10,000 docs and daily refreshes — now you're at dollars per day
> - The `AVG_TOKENS_PER_DOC` number is your key multiplier: `new_docs_per_day × avg_tokens × cost_per_token = daily_cost`
>
> **Rule of thumb:** If `avg_tokens_per_doc` is high (>2,000), consider chunking documents before embedding. If `total_tokens` for a full refresh exceeds your budget, that's your signal to implement incremental processing.

> :bulb: **Scale planning:** 8 docs is small. But if this were a chain with 500 menu items, 50 locations, and daily updates, those tokens add up. Use incremental logic (only process new/changed documents) and monitor token consumption over time. The numbers from this query are your baseline — multiply by your expected growth rate to forecast monthly costs.

---
### Building the automated pipeline — setup: a stage and a simulated new file
With the manual pieces built and costed, the rest of this section assembles them into **one automated pipeline** that processes new files on its own. First we need a new file for it to find. We'll:

To prove the pipeline Task actually processes new files, we need a new PDF that isn't already in the knowledge base. We'll:
1. Create a stage in your database to hold new uploads
2. Generate a simple PDF for a new drink (`matcha_latte`) using Python
3. Upload it to your stage

When the Task runs, it reads from **both** the shared stage (existing drinks) and **your stage** (new uploads). This simulates what happens in production when new content arrives.

> :bulb: **Why upload before creating the Task?** We upload the file first so the Task has something to find on its very first run — saving you an extra 2-minute wait. In production, the order is reversed: the automation is deployed first, then new files arrive over time. The end result is the same — the Task's incremental logic processes whatever is new regardless of when it was uploaded.

> :bulb: **Why `ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')`?** Cortex AI file functions — `AI_PARSE_DOCUMENT`, and the multimodal `AI_COMPLETE` over `TO_FILE`, and `AI_TRANSCRIBE` — **require** the stage holding the media to use server-side encryption. `SNOWFLAKE_SSE` means Snowflake manages the encryption keys and encrypts the files at rest, and it's what makes the staged files readable by these functions. A stage created without SSE will fail when an AI function tries to read from it. (The shared `DOCS_STAGE` you parsed from earlier is already SSE-encrypted; here you set it explicitly because you own this stage.)

In [ ]:
-- Create a stage with server-side encryption (required for AI_PARSE_DOCUMENT compatibility)
CREATE OR REPLACE STAGE {{my_db}}.RAW.COFFEE_DOCS_STAGE
  DIRECTORY = (ENABLE = TRUE)
  ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')
  COMMENT = 'Student stage for new coffee PDF uploads';


The cell below generates a small PDF for a brand-new drink (`matcha_latte`) entirely in Python — no external libraries — and uploads it to the stage you just created. The PDF-building code is verbose and not the point of this lab, so it's collapsed to show only its output. Just run it: you should see a confirmation that `matcha_latte.pdf` was uploaded. This is the "new file" your automated pipeline will discover.

In [ ]:
# Generate a test PDF for a new drink and upload to the student stage
import os

# Content for our new drink
content = """Matcha Latte

Matcha latte combines ceremonial-grade matcha green tea powder with steamed milk. The matcha is whisked with a small amount of hot water to form a paste, then combined with steamed milk and optionally sweetened. It has an earthy, slightly bitter flavor balanced by the creaminess of the milk. Typically served hot or iced in 12-16 oz cups. Contains approximately 70mg of caffeine per serving (less than espresso but more than regular green tea). The caffeine release is slower due to L-theanine, providing calm alertness without jitters. Popular variations include vanilla matcha latte and iced matcha latte with oat milk."""

# Build a minimal valid PDF (no external libraries needed)
def create_simple_pdf(text, filepath):
    """Create a minimal but valid PDF file from plain text."""
    lines = text.split('\n')
    # Use Tm (text matrix) for absolute positioning — avoids the cumulative Td offset bug
    stream_content = b'BT\n/F1 10 Tf\n'
    y = 750
    for line in lines:
        if y < 50:
            break
        safe_line = line.replace('(', '\\(').replace(')', '\\)').encode('latin-1', errors='replace')
        stream_content += f'1 0 0 1 72 {y} Tm\n'.encode() + b'(' + safe_line + b') Tj\n'
        y -= 14
    stream_content += b'ET\n'
    
    # Object 1: Catalog
    obj1 = b'1 0 obj\n<< /Type /Catalog /Pages 2 0 R >>\nendobj\n'
    # Object 2: Pages
    obj2 = b'2 0 obj\n<< /Type /Pages /Kids [3 0 R] /Count 1 >>\nendobj\n'
    # Object 3: Page
    obj3 = b'3 0 obj\n<< /Type /Page /Parent 2 0 R /MediaBox [0 0 612 792] /Contents 4 0 R /Resources << /Font << /F1 5 0 R >> >> >>\nendobj\n'
    # Object 5: Font
    obj5 = b'5 0 obj\n<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>\nendobj\n'
    # Object 4: Content stream
    obj4 = f'4 0 obj\n<< /Length {len(stream_content)} >>\nstream\n'.encode() + stream_content + b'endstream\nendobj\n'
    
    # Assemble PDF
    header = b'%PDF-1.4\n'
    body = obj1 + obj2 + obj3 + obj4 + obj5
    xref_offset = len(header) + len(body)
    xref = b'xref\n0 6\n0000000000 65535 f \n'
    offset = len(header)
    for obj in [obj1, obj2, obj3, obj4, obj5]:
        xref += f'{offset:010d} 00000 n \n'.encode()
        offset += len(obj)
    trailer = f'trailer\n<< /Size 6 /Root 1 0 R >>\nstartxref\n{xref_offset}\n%%EOF'.encode()
    
    with open(filepath, 'wb') as f:
        f.write(header + body + xref + trailer)

# Generate the PDF
tmp_path = '/tmp/matcha_latte.pdf'
create_simple_pdf(content, tmp_path)

# Upload to stage
session.file.put(
    tmp_path,
    f'@{my_db}.RAW.COFFEE_DOCS_STAGE',
    auto_compress=False,
    overwrite=True
)

print(f'Uploaded matcha_latte.pdf to @{my_db}.RAW.COFFEE_DOCS_STAGE')
print(f'File size: {os.path.getsize(tmp_path)} bytes')

In [ ]:
-- Refresh the directory metadata, then verify the file is on stage
ALTER STAGE {{my_db}}.RAW.COFFEE_DOCS_STAGE REFRESH;

SELECT RELATIVE_PATH, SIZE, LAST_MODIFIED
FROM DIRECTORY('@{{my_db}}.RAW.COFFEE_DOCS_STAGE');


---
## :robot_face: CoCo Assist — Build the Full Automated Pipeline

This is where everything comes together. You've manually built each piece of the pipeline:
1. **Parse** PDFs into text (Section 1)
2. **Classify** content by category (Section 2)
3. **Index** via Cortex Search Service (Section 3 — auto-refreshes)

And you just created a stage with a new test PDF (`matcha_latte.pdf`) that hasn't been processed yet.

Now your team lead asks: *"Can you automate this so the pipeline runs end-to-end whenever new PDFs arrive — from either the shared stage or our upload stage?"*

Instead of writing this yourself, let's use **CoCo** to generate the SQL. This is the real-world workflow: you describe the WHAT, CoCo handles the HOW.

### Set **Claude Opus 4.8 (or later)** as the selected model.

- In the Cortex Code chat panel select **Claude Opus 4.8 (or later)** as the model to use for generation. 

### 🔴 ACTION REQUIRED: Change Your Model

Select **`Claude Opus 4.8 (or later)`** in the Cortex Code panel now.

Note that this model family is described in the selector as `Most capable for ambitious work` and that is exactly what we are undertaking in the following example.

![CoCo model selector](images/sfp_select_claude_opus.png)

**What this prompt asks CoCo to build (in plain language):**

A scheduled job (Task) that runs every 2 minutes and performs the complete pipeline, reading from **two stages**:
- The shared stage (`@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE/coffee_kb/`) — existing menu PDFs
- Your upload stage (`@{{my_db}}.RAW.COFFEE_DOCS_STAGE`) — new PDFs you add

For each new file found on either stage, it:
- Parses them into text using `AI_PARSE_DOCUMENT`
- Classifies each new document using `AI_CLASSIFY`
- The Cortex Search Service automatically picks up the new rows on its next refresh — **no embedding step needed on your side.** Cortex Search embeds the indexed column for you; that's the whole point of using it instead of managing vectors yourself.
> :bulb: **On-demand vs. scheduled:** We create this Task with **no schedule** and trigger it once with `EXECUTE TASK` — perfect for a lab, and for pipelines you run manually. In production you'd typically add a `SCHEDULE` (hourly, daily) or make it stream-triggered so it fires automatically when new files land. Because it has no schedule, this Task will never run on its own — there's nothing to suspend afterward.

**Jargon decoder:**
| Term in the prompt | What it means |
|---|---|
| `Snowflake Task` | A scheduled job that runs SQL on a timer (like a cron job) |
| (no `SCHEDULE`) | The Task has no timer — it runs only when you explicitly trigger it with `EXECUTE TASK` |
| Two INSERT blocks | One per stage — each checks for new files independently |
| `AI_PARSE_DOCUMENT` | Extracts text from PDF files |
| `WHERE ... NOT IN (SELECT source_file ...)` | The incremental pattern — only parse files that aren't already in the table |
| `EXECUTE TASK` | Triggers a single, on-demand run of the Task right now |

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">/snowflake-tasks write a Snowflake Task called {{my_db}}.RAW.COFFEE_PIPELINE_TASK that uses warehouse {{my_wh}} and has NO schedule (it will be run on demand). It should read from TWO stages: 1a) Find PDF files in @PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE/coffee_kb/ that aren't already in {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE (compare by source_file column), parse with AI_PARSE_DOCUMENT in LAYOUT mode, and INSERT. 1b) Do the same for @{{my_db}}.RAW.COFFEE_DOCS_STAGE (all .pdf files). Then: 2) Classify any rows where category IS NULL using AI_CLASSIFY with categories ['Hot Drink', 'Cold Drink', 'Both Hot and Cold'], storing the result using :labels[0]::STRING. Use separate INSERT statements for each stage (not UNION), then an UPDATE for classification. Do NOT generate embeddings — the Cortex Search Service embeds and indexes new rows automatically. After creating the Task, include an EXECUTE TASK {{my_db}}.RAW.COFFEE_PIPELINE_TASK statement to run it once on demand. The Task must stay suspended (do NOT add RESUME) so it never runs on a schedule. Do NOT execute the code — just generate it so I can copy and paste it myself.</pre>

**What to do with the result:**
> 1. Copy CoCo's generated SQL
> 2. Paste it into the empty cell below
> 3. Run it
> 4. If it errors, paste the error message back to CoCo — it will fix it
> 5. The `EXECUTE TASK` line runs the pipeline once, immediately
> 6. Run the verification cells to confirm 9 rows (including the new drink)

**A note on prompting:** Prompts are iterative. You may not get perfect output on the first try — that's normal and expected. If CoCo generates something that doesn't quite work, refine your prompt, add more detail, or paste the error back. This back-and-forth is how real engineering with AI assistants works. If you get stuck or want to see what a working solution looks like, expand the **"Resolution — Pipeline Task"** cell below (it's collapsed by default).

**Why this is the culmination:** This single Task ties the pipeline together — parse and classify across multiple data sources with an incremental `NOT IN` guard — and creates a production-ready ingestion flow. The Cortex Search Service and Agent you built earlier automatically benefit from the new rows (Search re-embeds and re-indexes on its own refresh), so there is no additional work on that side.

This iterate-with-CoCo workflow is how engineers build automation without memorizing Task syntax.

In [ ]:
-- Paste CoCo's suggestion on the blank line below, then run it


<details>
<summary>🔧 <b>Resolution — Pipeline Task</b> (click to expand a working solution, then copy it into the empty cell above)</summary>

```sql
-- Complete pipeline Task. No schedule, no RESUME — created suspended and run once on demand.
CREATE OR REPLACE TASK {{my_db}}.RAW.COFFEE_PIPELINE_TASK
  WAREHOUSE = {{my_wh}}
AS
BEGIN
  -- Refresh directory metadata so newly uploaded files are visible
  ALTER STAGE {{my_db}}.RAW.COFFEE_DOCS_STAGE REFRESH;

  -- Step 1a: new PDFs from the shared stage
  INSERT INTO {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE (drink_id, source_file, content)
  SELECT
    REPLACE(SPLIT_PART(d.RELATIVE_PATH, '/', -1), '.pdf', '') AS drink_id,
    d.RELATIVE_PATH AS source_file,
    AI_PARSE_DOCUMENT(
      TO_FILE('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE', d.RELATIVE_PATH),
      {'mode': 'LAYOUT'}
    ):content::STRING AS content
  FROM DIRECTORY('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE') d
  WHERE d.RELATIVE_PATH LIKE 'coffee_kb/%.pdf'
    AND d.RELATIVE_PATH NOT IN (
      SELECT source_file FROM {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE WHERE source_file IS NOT NULL
    );

  -- Step 1b: new PDFs from your upload stage
  INSERT INTO {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE (drink_id, source_file, content)
  SELECT
    REPLACE(SPLIT_PART(d.RELATIVE_PATH, '/', -1), '.pdf', '') AS drink_id,
    d.RELATIVE_PATH AS source_file,
    AI_PARSE_DOCUMENT(
      TO_FILE('@' || '{{my_db}}' || '.RAW.COFFEE_DOCS_STAGE', d.RELATIVE_PATH),
      {'mode': 'LAYOUT'}
    ):content::STRING AS content
  FROM DIRECTORY('@{{my_db}}.RAW.COFFEE_DOCS_STAGE') d
  WHERE d.RELATIVE_PATH LIKE '%.pdf'
    AND d.RELATIVE_PATH NOT IN (
      SELECT source_file FROM {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE WHERE source_file IS NOT NULL
    );

  -- Step 2: classify any rows missing a category
  UPDATE {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE
  SET category = AI_CLASSIFY(content, ['Hot Drink', 'Cold Drink', 'Both Hot and Cold']):labels[0]::STRING
  WHERE category IS NULL;
END;

-- Run the pipeline once, on demand. The Task stays suspended (no schedule, no RESUME).
EXECUTE TASK {{my_db}}.RAW.COFFEE_PIPELINE_TASK;
```

</details>

### Verify the pipeline ran

The `EXECUTE TASK` line runs the pipeline once, right away — you don't need to wait for a schedule. Give it a few seconds, then run the cells below to confirm it executed.

**How to tell if the pipeline processed anything:**

1. **Check Task History cell** — Look for `state = SUCCEEDED`. This confirms the Task ran without errors.
   - If you see `EXECUTING` — the Task is still running. Give it a few seconds and **re-run the Check Task History cell**.
   - If you see no rows at all — the `EXECUTE TASK` hasn't registered yet. Wait a few seconds and re-run.
   - Once you see `SUCCEEDED`, move on to the verification cells below.
2. **Verify Pipeline Results cell** — Shows every row with `is_classified` and `parsed_at`. 
   - If new files were processed, you'll see rows with **recent `PARSED_AT` timestamps** (within the last few minutes)
   - If all files were already processed earlier in the lab (which is expected!), the Task still ran successfully — it just found nothing new to do. That's the incremental pattern working correctly: no new PDFs on stage = no new rows inserted.

> :warning: **Timing note:** The Task's steps (insert → classify → embed) run **sequentially** within a single execution — they are NOT asynchronous. However, if you run the verification cells *while* the Task is still executing (state = `EXECUTING`), you may see a new row without its classification or embedding yet. This doesn't mean embedding failed — it means the Task was mid-execution when you checked. Always wait for `SUCCEEDED` in the Task History before verifying results.

> :bulb: **"The Task succeeded but didn't insert anything"** is a valid outcome. It means the `NOT IN` check correctly detected that all PDFs are already in the table. In production, this is what most runs look like — the Task only does work when new files actually arrive.

Because the Task has no schedule, there's nothing to clean up — it only ran the one time you triggered it and will stay idle until you `EXECUTE TASK` again.

In [ ]:
-- Check Task execution history (should show SUCCEEDED)
SELECT name, state, scheduled_time, completed_time, error_message
FROM TABLE({{my_db}}.INFORMATION_SCHEMA.TASK_HISTORY(
  TASK_NAME => 'COFFEE_PIPELINE_TASK',
  SCHEDULED_TIME_RANGE_START => DATEADD(hour, -1, CURRENT_TIMESTAMP())
))
ORDER BY scheduled_time DESC
LIMIT 5;


In [ ]:
-- Verify all rows are fully processed (classified)
-- If the Task processed new files, you'll see rows with recent PARSED_AT timestamps
SELECT
  drink_id,
  category IS NOT NULL AS is_classified,
  parsed_at
FROM {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE
ORDER BY parsed_at DESC;


### Confirm the new drink was processed

The matcha latte PDF you generated earlier contains unique content that no other drink in the knowledge base has — terms like "matcha", "green tea", "L-theanine", and "ceremonial-grade". If the pipeline processed it correctly, you should see a new row with:
- `drink_id` = `matcha_latte`
- `category` assigned (likely "Hot Drink" or "Both Hot and Cold")
- Content containing matcha-specific terms

(No embedding column to check — Cortex Search handles embedding the new row for you on its next refresh.)

This is your proof that the full pipeline worked end-to-end: **a raw PDF you created in Python → was parsed, classified, and embedded by the automated Task → is now queryable in your knowledge base.**

In [ ]:
-- Prove the new drink was processed by the pipeline (filter by drink_id, not content)
SELECT
  drink_id,
  category,
  parsed_at,
  LEFT(content, 150) AS content_preview
FROM {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE
WHERE drink_id = 'matcha_latte';


### Make the new drink searchable — force a refresh

The row is in the knowledge base, but it isn't in the **Cortex Search Service** yet. The service re-indexes on its own within its `TARGET_LAG` window (you set it to **1 hour**), so in normal operation `matcha_latte` would become searchable within the hour — no action needed.

When you need new content available *immediately* — a demo, a smoke test, or a document you just added — trigger a manual refresh with `ALTER CORTEX SEARCH SERVICE ... REFRESH`. It reindexes on the spot instead of waiting for the lag window. Run the refresh cell, then query the service and confirm `matcha_latte` now comes back.

In [ ]:
-- Force the search service to reindex now, instead of waiting out the 1-hour TARGET_LAG
ALTER CORTEX SEARCH SERVICE {{my_db}}.RAW.COFFEE_SEARCH_SVC REFRESH;


In [ ]:
-- Confirm the new drink is now searchable (matcha_latte should appear in the results)
SELECT
    f.value:"drink_id"::STRING AS drink_id,
    LEFT(f.value:"content"::STRING, 200) AS content_preview
FROM TABLE(FLATTEN(
    PARSE_JSON(SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        '{{my_db}}.RAW.COFFEE_SEARCH_SVC',
        '{
            "query": "matcha green tea latte",
            "columns": ["drink_id", "content"],
            "limit": 3
        }'
    )):results
)) f;


---
## Verify
If your lab ran successfully, you should have:
- `{{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE` with **9 rows** (8 original + matcha_latte), all classified
- `{{my_db}}.RAW.COFFEE_SEARCH_SVC` Cortex Search Service (active)
- `{{my_db}}.RAW.{{user}}_COFFEE_MENU_AGENT` Cortex Agent (responding to questions)
- `{{my_db}}.RAW.COFFEE_PIPELINE_TASK` Task (created, run once on demand, no schedule)
- `{{my_db}}.RAW.COFFEE_DOCS_STAGE` Stage with your test PDF
- (Section 3) showing logical drink relationships

---
### Check your understanding 🧠
A few questions on this lab's key themes. Run each quiz cell below, choose an answer, and review the feedback.

In [ ]:
from resources.knowledgecheck import data_engineer_quiz as quiz
quiz.display_question("q1")

In [ ]:
from resources.knowledgecheck import data_engineer_quiz as quiz
quiz.display_question("q2")

In [ ]:
from resources.knowledgecheck import data_engineer_quiz as quiz
quiz.display_question("q3")

In [ ]:
from resources.knowledgecheck import data_engineer_quiz as quiz
quiz.display_question("q4")

---
### ❄️ Key Takeaways
- **DIRECTORY + AI_PARSE_DOCUMENT** = scalable document ingestion pipeline
- **AI_CLASSIFY** enriches parsed data with categorical metadata
- **Cortex Search Service** provides semantic search over your document collection
- **Cortex Agent** combines search + LLM for conversational Q&A
- **AI_EMBED** is the embedding primitive — useful directly for "find similar" (Section 3), and used under the hood by Cortex Search
- **Multimodal AI** (gemini) can analyze video content in the same pipeline
- Production pattern: Parse → Classify → Index (Search, which embeds for you) → Serve (Agent)

**What's Next — The Other Personas:**
- The **Data Analyst** notebook shows the Streamlit prototype kiosk that proves this concept to stakeholders
- The **Platform Admin** notebook covers governance (PII redaction, content moderation, cost monitoring) and deploys the production kiosk via CoWork
- Together, all three personas create a complete, governed, customer-facing AI system

### Notebook completion. 🏁
You're done — your notebook is saved automatically.

> 💡 **Moving on to another lab?** Shut down this notebook's **kernel** — click **Connected** at the top → **Shut down kernel**. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for the next lab. Your service stays running, ready in seconds.

> 💡 **Finished for the day?** **Suspend your service** — click **Connected** → hover your service name → **Suspend**. This stops compute; it resumes automatically next time you run a cell.